In [5]:
import numpy as np
import pandas as pd
from pylab import plt, mpl
from sklearn.metrics import accuracy_score
import os
import papermill as pm
import talib as ta
from sklearn.model_selection import TimeSeriesSplit

frequency = "1d"

# Cargar los datos para esta frecuencia
'''BTCUSDTd = pd.read_csv('BTCUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')'''
BTCUSDT4h = pd.read_csv('BTCUSDT_4h_01-01-2016_01-01-2025.csv', index_col='timestamp')
'''BTCUSDTh = pd.read_csv('BTCUSDT_1h_01-01-2016_01-01-2025.csv', index_col='timestamp')'''
data = pd.DataFrame() 

'''data['BTCUSDTd'] = BTCUSDTd[['close']]'''
data['BTCUSDT4h'] = BTCUSDT4h[['close']]
'''data['BTCUSDTh'] = BTCUSDTh[['close']]'''
data.dropna(inplace=True)
data.head()

,BTCUSDT4h
timestamp,
2017-08-17 04:00:00,4349.99
2017-08-17 08:00:00,4427.30
2017-08-17 12:00:00,4352.34
2017-08-17 16:00:00,4325.23
2017-08-17 20:00:00,4285.08


Función para guardar los datos. Hace un archivo por cada frecuencia. Guarda en cada línea el modelo que se ha empleado, el activo, accuracy e in/out-sample.

In [6]:
def save_results(model, ric, acc, sample, frequency=frequency):
    # Verificar si el archivo ya existe
    file_name = f'accuracy_results_{frequency}_charac.csv'

    # Si el archivo existe, leer los datos previos, si no, crear un nuevo DataFrame vacío
    if os.path.exists(file_name):
        df_results = pd.read_csv(file_name)
    else:
        df_results = pd.DataFrame(columns=['Model', 'Asset', 'Accuracy', 'IN/OUT Sample'])

    # Agregar la nueva fila con los resultados
    new_row = pd.DataFrame([[model, ric, acc, sample]], columns=['Model', 'Asset', 'Accuracy', 'IN/OUT Sample'])
    df_results = pd.concat([df_results, new_row], ignore_index=True)

    # Guardar los resultados acumulados
    df_results.to_csv(file_name, index=False) 

Creamos las características que usaremos para hacer el aprendizaje ahora y las retardamos.

In [7]:
def add_lags(data, ric, lags, window=30):
    cols = []
    df = pd.DataFrame(data[ric])
    df.dropna(inplace=True)
    df['r'] = np.log(df / df.shift()) #retornos
    df['sma'] = df[ric].rolling(window).mean()  #media movil de la ventana
    df['min'] = df[ric].rolling(window).min() #mínimo de la ventana
    df['max'] = df[ric].rolling(window).max() #máximo de la ventana
    df['mom'] = df[ric].pct_change(window) #momentum de la ventana
    df['vol'] = df['r'].rolling(window).std() #volatilidad de la ventana
    df['rsi'] = ta.RSI(df[ric], timeperiod=window) #rsi de la ventana
    df['atr'] = ta.ATR(df[ric], df[ric], df[ric], timeperiod=window) #atr de la ventana
    df.dropna(inplace=True)
    df['d'] = np.where(df['r'] > 0, 1, 0) # columna binaria, 0 si los precios bajarán, 1 si subirán
    features = [ric, 'r', 'd', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr']
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    df.dropna(inplace=True)
    return df, cols

lags = 1

dfs = {}
for ric in data:
    df, cols = add_lags(data, ric, lags)
    dfs[ric] = df.dropna(), cols

In [10]:
from sklearn.neural_network import MLPClassifier

for ric in data:
    model = MLPClassifier(hidden_layer_sizes=[512],
                        random_state=100,
                        max_iter=1000,
                        early_stopping=True,
                        validation_fraction=0.15,
                        shuffle=False)
    df, cols = dfs[ric]
    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()
    model.fit(df[cols], df['d'])
    pred = model.predict(df[cols])
    acc = accuracy_score(df['d'], pred)
    print(f'IN-SAMPLE | {ric:7s} | acc={acc:.4f}')
    save_results('MLPClassifier', ric, acc, "IN-SAMPLE")



IN-SAMPLE | BTCUSDT4h | acc=0.5436


In [11]:
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential

np.random.seed(100)
tf.random.set_seed(100)

def create_model(problem='regression'):
    model = Sequential()
    model.add(Dense(512, input_dim=len(cols),
                    activation='relu'))
    if problem == 'regression':
        model.add(Dense(1, activation='linear'))
        model.compile(loss='mse', optimizer='adam')
    else:
        model.add(Dense(1, activation='sigmoid'))
        model.compile(loss='binary_crossentropy', optimizer='adam')
    return model

for ric in data:
    model = create_model('classification')
    df, cols = dfs[ric]
    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()
    model.fit(df[cols], df['d'], epochs=50, verbose=False)
    pred = np.where(model.predict(df[cols]) > 0.5, 1, 0)
    acc = accuracy_score(df['d'], pred)
    print(f'IN-SAMPLE | {ric:7s} | acc={acc:.4f}')
    save_results('classification', ric, acc, "IN-SAMPLE")

c:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


504/504 ━━━━━━━━━━━━━━━━━━━━ 0s 737us/step
IN-SAMPLE | BTCUSDT4h | acc=0.5535


In [13]:
df = df[cols+['d']]
df

,BTCUSDT4h_lag_1,r_lag_1,d_lag_1,sma_lag_1,min_lag_1,max_lag_1,mom_lag_1,vol_lag_1,rsi_lag_1,atr_lag_1,d
timestamp,,,,,,,,,,,
2017-08-22 08:00:00,-1.055646,-1.095481,-1.031936,-1.042074,-1.043184,-1.041880,-1.632166,0.181322,-2.050658,-0.873643,1
2017-08-22 12:00:00,-1.052543,1.138223,0.968992,-1.042926,-1.043184,-1.045164,-1.629733,0.182832,-1.730445,-0.870824,0
2017-08-22 16:00:00,-1.052917,-0.150493,-1.031936,-1.043677,-1.043184,-1.045691,-1.472193,0.159393,-1.750507,-0.877163,1
2017-08-22 20:00:00,-1.047200,2.059655,0.968992,-1.044195,-1.043184,-1.045691,-1.056465,0.336072,-1.198458,-0.865546,1
2017-08-23 00:00:00,-1.044670,0.883461,0.968992,-1.044567,-1.043184,-1.045691,-0.797086,0.364884,-0.974722,-0.864902,1
...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 08:00:00,2.984907,0.231562,0.968992,3.081472,3.159805,3.012170,-0.707593,-0.361228,-0.835675,2.247013,1
2024-12-31 12:00:00,3.047148,0.932977,0.968992,3.079249,3.159805,3.012170,-0.294900,-0.391440,-0.484050,2.342232,1
2024-12-31 16:00:00,3.100651,0.789262,0.968992,3.078290,3.159805,3.012170,-0.190391,-0.360440,-0.205063,2.405254,0


Hacemos una función que entrene el modelo, lo valide utilizando walk-forward y calcule el accuracy.

In [16]:
def walk_forward_fit_test(model_class, freq, model_params={}):
    # que el tamaño de ventana se modifique con la frecuencia
    if freq == '1h': i=24
    elif freq == '4h': i=6
    else: i=1
    
    for ric in data:
        df, cols = dfs[ric]
        df = df[cols+['d']]
        n = int(round(len(df)/(90*i)))
        s = int(round(n*0.35))
        tscv = TimeSeriesSplit(n_splits=n, test_size=s, gap=1)
        results = []
        for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
            train, test = df.iloc[train_idx], df.iloc[test_idx]
            
            X_train, y_train = train.drop(columns=['d']), train['d']
            X_test, y_test = test.drop(columns=['d']), test['d']

            # Normalizar usando solo datos de entrenamiento
            mean, std = X_train.mean(), X_train.std()
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std 

            # Crear un modelo nuevo en cada fold para evitar sobreajuste
            model = model_class(**model_params)
            model.fit(X_train, y_train)

            # Predicción y evaluación
            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

        # Guardar e imprimir resultados
        avg_acc = np.mean(results)
        print(f'OUT-OF-SAMPLE | {ric:7s} | acc={avg_acc:.4f}')
        save_results(model_class.__name__, ric, avg_acc, "OUT-SAMPLE")


Modelo MLP Classifier

In [17]:
from sklearn.neural_network import MLPClassifier        

walk_forward_fit_test(MLPClassifier, '4h', {"hidden_layer_sizes":[512],
                            "random_state":100,
                            "max_iter":1000,
                            "early_stopping":True,
                            "validation_fraction":0.15,
                            "shuffle":False})

OUT-OF-SAMPLE | BTCUSDT4h | acc=0.5667


Modelo Bagging Classifier

In [ ]:
from sklearn.ensemble import BaggingClassifier

base_estimator = MLPClassifier(hidden_layer_sizes=[256],
                            random_state=100,
                            max_iter=1000,
                            early_stopping=True,
                            validation_fraction=0.15,
                            shuffle=False) 


walk_forward_fit_test(BaggingClassifier, {"base_estimator":base_estimator,
                            "n_estimators":35,
                            "max_samples":0.25,
                            "max_features":0.5,
                            "bootstrap":False,
                            "bootstrap_features":True,
                            "n_jobs":8,
                            "random_state":100})

c:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
c:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
c:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
c:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
c:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: Futu

OUT-OF-SAMPLE | BTCUSDT4h | acc=0.9823


C:\Users\raque\AppData\Local\Temp\ipykernel_23176\3694455987.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, new_row], ignore_index=True)
